<a href="https://colab.research.google.com/github/otoperalias/teaching/blob/TallerUTE_AnalisisCuanti/UTE_TallerAnalisisCuanti_2024_agosto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://github.com/otoperalias/teaching/blob/TallerUTE_AnalisisCuanti/material/6.%20M%C3%89TODOS%20CUANTITATIVOS%20WORKSHOP%20(1).jpg?raw=true" alt="drawing" width="900"/>



## Taller de Análisis de Datos Cuantitativos

### Maestría en Urbanismo (UTE, Ecuador).

Instructor: [Daniel Oto-Peralías](https://otoperalias.github.io/)
_________________________

El objetivo del taller es aprender a explotar una encuesta o un censo a través de programas estadísticos de análisis cuantitativo.

Usaremos el nuevo [Censo de Población y Vivienda de Ecuador (2022)](https://www.censoecuador.gob.ec/) para analizar las características de las viviendas de la provincia de Loja. La tarea a realizar será la creación de un indicador de calidad de las viviendas.

**Vamos a necesitar el siguiente material:**
1. Los (micro)datos del censo.  
Corresponden a los registros de cada vivienda. Todos los microdatos están disponibles [aquí](https://www.censoecuador.gob.ec/data-censo-ecuador/). Los ficheros corresponden al conjunto del país y, por tanto, tienen un gran tamaño y su procesamiento lleva algún tiempo. Para facilitar el acceso a los datos de vivienda de Loja, los he extraído previamente y son accesibles [**aquí**](https://github.com/otoperalias/teaching/blob/TallerUTE_AnalisisCuanti/material/censo22/viv_loja.csv). En [este notebook](https://github.com/otoperalias/teaching/blob/TallerUTE_AnalisisCuanti/material/censo22/Filtrado_tabla_viviendas.ipynb) se muestran todos los pasos para la descarga y extracción de los datos (es una tarea sencilla, pero supone cierto tiempo debido al tamaño de los ficheros).
2. El [cuestionario del censo](https://www.censoecuador.gob.ec/wp-content/uploads/2023/11/Cuestionario_censal.pdf), donde podemos ver el significado de los códigos numéricos que aparecen en los datos.
3. El diccionario de datos del censo, para conocer el significado de cada variable. Se puede obtener a través de [este enlace](https://www.censoecuador.gob.ec/wp-content/uploads/2024/02/4_DICCIONARIO_DE_VARIABLES_CPV_2022.xlsx).
4. Además, es útil tener a mano la infografía con la descripción de los resultados del censo para la provincia de Loja. Disponible [aquí](https://www.censoecuador.gob.ec/ecuadormap/), clicando en la provincia de Loja. También es útil consultar la *Guía de usuario*, disponible también [aquí](https://www.censoecuador.gob.ec/data-censo-ecuador/).
5. Por ultimo, descargamos las capas de *Cartografía*, para poder representar geográficamente la información. Está disponible [aquí](https://www.ecuadorencifras.gob.ec/documentos/web-inec/capa/CapaSectores.zip), comprimido en formato geopackage de QGIS. Como el fichero a descargar corresponde al conjunto de país y, por tanto, pesa mucho, he procesado los datos para extraer solamente la provincia de Loja, y así no invertir tanto tiempo en esto durante el taller. La cartografía para Loja (formato .shp) está disponible [aquí](https://github.com/otoperalias/teaching/blob/TallerUTE_AnalisisCuanti/material/censo22/sect_loja_1.shp.zip) y [aquí](https://github.com/otoperalias/teaching/blob/TallerUTE_AnalisisCuanti/material/censo22/sect_loja_2.shp.zip) (dividida en dos ficheros) y el notebook que procesa los datos originales [aquí](https://github.com/otoperalias/teaching/blob/TallerUTE_AnalisisCuanti/material/censo22/Procesam_cartografia.ipynb).

**Sobre el programa informático:**

* En este taller usamos [**Google Colab**](https://colab.research.google.com/?hl=es), que es un *notebook* virtual desde el que podemos usar **Python**.
* La librería principal que vamos a usar es **Pandas**, una "paquete" especializado en el procesamiento y análisis de datos cuantitativos.
* La gran ventaja de Python-Pandas frente a otras alternativas (SPSS/Stata/etc.) es su carácter gratuito y la gran cantidad de recursos de ayuda que existe en Internet, debido a su enorme comunidad de usuarios.

## 1. Importamos las librerías que vamos a utilizar

Para usar el paquete de **Pandas** y poder visualizar los datos, tenemos que importar estas librerías:

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Importamos los datos


Para importar los datos, en primer lugar, tenemos que descargar los datos en nuestro ordenador y subirlos a Google Colab:
1. Clicamos en el icono de carpeta que hay en la parte superior de la barra de la izquierda (📁) y entonces clicamos en el primer icono de upload.
2. Se abre una ventana para seleccionar el archivo que queremos subir y seleccionamos la carpeta "viv_loja".

Una vez que tenemos subido el fichero y que aparece como tal ("viv_loja.csv") en la barra izquierda, podemos importarlo:

In [ ]:
viv=pd.read_csv("viv_loja.csv")

Para visualizar la tabla, escribimos el nombre que le hemos dado

In [ ]:
viv

Como se observa, esta tabla de datos (```dataframe```) tiene 199013 filas y 36 columnas. Cada fila contiene los datos censales de una vivienda.  
En los ```dataframe``` las filas se identifican con un ```index```, que generalmente es único para cada fila. Las columnas se identifican con un nombre. No debe confundirse el index de la tabla con una columna. Es decir, el index no es la primera columna.

<img src="https://github.com/otoperalias/teaching/blob/TallerUTE_AnalisisCuanti/material/df_structure.jpg?raw=true" alt="drawing" width="550"/>

## 3. Exploración inicial

Para conocer las columnas que contiene la tabla, escribimos el siguiente código:

In [ ]:
viv.columns

Nótese que necesitamos el diccionario de datos del censo para conocer el significado de cada columna. Por otra parte, para saber el significado de los diferentes valores que toman las variables, necesitamos el cuestionario del censo. Ambos documentos se mencionan al comienzo de este notebook.

Para obtener el número de filas de la tabla (o sea, su longitud) escribimos lo siguiente:

In [ ]:
len(viv)

En la infografía con los resultados para Loja, vemos que en esta provincia hay 143.741 viviendas ocupadas.  
Podemos comprobar si los datos de nuestra tabla nos dan la misma cifra. Para ello, aplicamos la función ```value_counts()```, que proporciona la distribución de frecuencias de una variable, sobre la variable V0201 ("condición de ocupación de vivienda particular").

In [ ]:
viv.V0201.value_counts()

Sumando las categorías 1 y 2 resultan 143.741 viviendas, exactamente la cifra anterior. Eso nos indica que los datos están cargados correctamente.

In [ ]:
138537+5204

También podemos calcular el porcentaje de viviendas según el área, urbana o rural. Para ello, usamos la variable AUR.

In [ ]:
# primero filtramos las viviendas ocupadas:
vivocup=viv.loc[viv['V0201']<=2].copy()

In [ ]:
# entonces, aplicamos la función value_counts(), con el argumento normalize=True, para obtener el tanto porcentual.
vivocup.AUR.value_counts(normalize=True)*100

Vemos de nuevo que es exactamente el valor del documento con la infografía de Loja.

**Nota sobre terminología**: las palabras *variable*, *campo* y *columna* se refieren a lo mismo. En inglés, *variable*, *field* y *column*.

Ahora vamos a representar dicha información en formato gráfico, como se hace en la infografía.

In [ ]:
vivocup.AUR.value_counts(normalize=True).multiply(100).plot.bar()

El siguiente bloque de código procesa ligeramente los datos para crear un gráfico más entendible. En concreto, creamos una nueva variable en la que asignamos a cada código numérico su etiqueta.

In [ ]:
# Procesamos los datos para mejorar el gráfico
vivocup['area']="" # Añadimos una nueva columna y en las líneas de código siguientes le damos contenido según los códigos de la columna AUR
vivocup.loc[vivocup['AUR']==1,'area']="URBANA"
vivocup.loc[vivocup['AUR']==2,'area']="RURAL"


Creamos el gráfico:

In [ ]:
fig,ax=plt.subplots()  # Aquí se indica que queremos crear una figura (fig) que contiene un gráfico (ax)
vivocup.area.value_counts(normalize=True).multiply(100).plot.bar() # Representar los porcentajes como gráfico de barras
ax.tick_params(axis='x', rotation=0) # Etiquetas del eje x en horizontal (rotadas 0 grados).
ax.set_ylabel("Porcentaje") # Título del eje y, donde se indica la unidad de medida
ax.set_xlabel("")  # No queremos título en el eje x, porque se sobre entiende.
ax.set_title("PORCENTAJE DE VIVIENDAS OCUPADAS SEGÚN ÁREA", weight='bold')  # Título de la figura
plt.show() # Comando para que se dibuje el gráfico.

## 4. Construcción del indicador de calidad de las viviendas

### 4.1. Diseño del indicador.


Nuestro objetivo es construir un indicador sintético (o compuesto) de calidad de las viviendas. Los indicadores o índices sintéticos se construyen para medir algo que no es directamente cuantificable. Por ejemplo, la altura de los edificios o el material del suelo sí es observable y cuantificable directamente, pero *la calidad* de las viviendas depende del criterio que fije el investigador. Por tanto, lo primero que hay que hacer es definir el concepto de calidad de la vivienda y, posteriormente, operacionalizarlo.  
**En general, para construir un indicador sintético hay que seguir estos pasos:**
1. **Conceptualización**: definir el fenómeno que se quiere medir.
2. **Dimensiones**: precisar las dimensiones del fenómeno a medir.  
3. **Indicadores parciales**: qué indicadores vamos a utilizar para medir cada dimensión.
4. **Variables simples**: las variables que van a formar parte de cada indicador parcial.
5. **Método de agregación**: decidir el método de agregación de las variables simples en indicadores parciales, de los indicadores parciales en dimensiones y de las dimensiones en el indicador sintético. La media aritmética es el método más sencillo e implica que los diferentes componentes tienen la misma importancia y se compensan perfectamente entre sí (es decir, un valor alto en uno compensa un valor bajo en otro). Otros métodos son, por ejemplo, la media ponderada, donde a cada componente se le asigna un peso diferente, y la media geométrica, que evita que los componentes sean directamente compensables entre sí.

Aplicado al indicador que queremos construir:
1.  Definimos la calidad de la vivienda como la calidad de los materiales, el estado de conservación y el nivel de salubridad.
2. Tenemos tres dimensiones: calidad de los materiales, estado de conservación y nivel de salubridad.
3. Vamos a usar los siguientes indicadores parciales:
 * Calidad de los materiales: material predominante en el techo (V03), paredes (V05) y piso (V07 ).
 * Estado de conservación: estado del techo (V04), paredes (V06) y suelo (V08).
 * Salubridad: suministro de agua por red pública (V10), conectado a alcantarillado (V11) y basura al carro corrector/contenedor (V14).
4. En el punto anterior se incluye entre paréntesis las variables del censo que vamos a utilizar.
5. El método de agregación será siempre, por sencillez, la media aritmética.

### 4.2. Variables simples.

#### 4.2.1. Calidad de materiales

In [ ]:
# Techo (1 si hormigón o teja; 0 en los demás casos)
vivocup['techo_mat']=0
vivocup.loc[((vivocup['V03']==1) | (vivocup['V03']==4)), 'techo_mat']=1

# Pared (1 si hormigón o ladrillo-bloque; 0 en los demás casos)
vivocup['pared_mat']=0
vivocup.loc[((vivocup['V05']==1) | (vivocup['V05']==2)), 'pared_mat']=1

# Piso (1 si madera tratada (duela, parquet...), baldosas-cerámica o marmol-marmetón; 0 en los demás casos)
vivocup['piso_mat']=0
vivocup.loc[(vivocup['V07']<=3), 'piso_mat']=1

# Echamos un vistazo a las variables creadas
vivocup[['techo_mat','pared_mat','piso_mat']].describe()

#### 4.2.2. Estado de conservación



In [ ]:
# Techo (1 si bueno, 0.5 regular y 0 malo)
vivocup['techo_est']=(3-vivocup['V04'])*0.5

# Pared (1 si bueno, 0.5 regular y 0 malo)
vivocup['pared_est']=(3-vivocup['V06'])*0.5

# Suelo (1 si bueno, 0.5 regular y 0 malo)
vivocup['piso_est']=(3-vivocup['V08'])*0.5

# Echamos un vistazo a las variables creadas
vivocup[['techo_est','pared_est','piso_est']].describe()

#### 4.2.3. Salubridad

In [ ]:
# Suministro de agua (1 si red pública o juntas de agua...; 0 el resto).
vivocup['agua']=0
vivocup.loc[(vivocup['V10']<=2) , 'agua']=1

# Alcantarillado (1 si red pública, 0 el resto)
vivocup['alcant']=0
vivocup.loc[(vivocup['V11']==1), 'alcant']=1

# Basura (1 si carro colector o contenedor, 0 el resto)
vivocup['basura']=0
vivocup.loc[(vivocup['V14']<=2) , 'basura']=1

# Echamos un vistazo a las variables creadas
vivocup[['agua','alcant','basura']].describe()

Podemos comprobar que las medias (que son proporciones) coinciden con los porcentajes de la infografía sobre Loja.

### 4.3. Construcción de los indicadores parciales

In [ ]:
# Calidad de materiales
vivocup["material"]=(vivocup['techo_mat']+vivocup['pared_mat']+vivocup['piso_mat'])/3
# Estado de conservación
vivocup["estado"]=(vivocup['techo_est']+vivocup['pared_est']+vivocup['piso_est'])/3
# Salubridad
vivocup["salu"]=(vivocup['agua']+vivocup['alcant']+vivocup['basura'])/3

# Echamos un vistazo a los indicadores parciales creados
vivocup[['material','estado','salu']].describe()

**Nota metodológica**: nótese que en este caso todas las variables son directamente agregables, ya que tienen escala de 0 a 1. Si las escalas fueran diferentes habría que estandarizar las variables para convertirlas a la misma escala. De lo contrario, las variables con una escala mayor (por ejemplo, de 0 a 100) pesarían más en el indicador.

### 4.4. Construcción del indicador sintético

In [ ]:
# Indicador sintético de calidad de las viviendas:
vivocup["calidad"]=(vivocup['material']+vivocup['estado']+vivocup['salu'])/3

# Echamos un vistazo al indicador sintético de calidad de las viviendas:
vivocup['calidad'].describe()

In [ ]:
# Distribución de frecuencias
sns.histplot(data=vivocup,x="calidad",stat="percent")

## 5. Agregaciones espaciales

Hasta ahora hemos estado usando datos individuales (a nivel de vivienda), pero con frecuencia nos interesa conocer los datos agregados de diferentes unidades geográficas, como la parroquia o el cantón.  
Vamos a agrupar las filas de la tabla por parroquias usando la función ```groupby()```.

Según el INEC, el código de la parroquia es la concatenación de los dos dígitos del código provincial, los dos del cantonal y los dos del parroquial. El código se proporciona en la columna PARROQ, por lo tanto, no es necesario crearlo.

In [ ]:
# Con el siguiente código obtenemos la lista de parroquias de Loja
vivocup.PARROQ.unique()

In [ ]:
# Vemos que hay 94
len(vivocup.PARROQ.unique())

Ahora ya podemos agrupar por parroquias usando la función ```groupby()```.

In [ ]:
columnas=['PARROQ','material', 'estado', 'salu', 'calidad']
viv_parroq=vivocup[columnas].groupby(by='PARROQ', as_index=False).mean()


Esta es la tabla agregada a nivel parroquial:

In [ ]:
viv_parroq

Obsérvese que la función ```mean()``` tiene sentido para agregar estas variables, pero téngase en cuenta que este no siempre es el caso.

Ahora vamos a echar un vistazo a la distribución de frecuencias a este nivel agregado:

In [ ]:
sns.histplot(data=viv_parroq,x="calidad",stat="percent")

## 6. Representación geográfica

A continuación vamos a representar los datos geográficamente. Para ello, tenemos que importar la cartografía de la provincia de Loja. Dicha capa, disponible en dos ficheros, la descargamos inicialmente y la debemos tener en nuestro ordenador. Debemos subir los dos ficheros a Google Colab, sin descomprimir.

A continuación, leemos los ficheros con la función de ```geopandas``` ```read_file()```

In [ ]:
# Importamos los dos ficheros
sectores1=gpd.read_file("sect_loja_1.shp.zip")
sectores2=gpd.read_file("sect_loja_2.shp.zip")
# Los unimos, con la función de concatenar
sectores=pd.concat([sectores1,sectores2])

Las divisiones geográficas vienen a nivel de sector, una división censal muy detallada, como podemos ver a continuación. Con ```geopandas```es muy fácil representar geográficamente:

In [ ]:
sectores.plot()

In [ ]:
len(sectores)

La representar la información geográficamente en parroquias, tenemos que generar una capa de parroquias. Esto es sencillo, simplemente hay que "disolver" por parroquias.

In [ ]:
parroq=sectores.dissolve("parroquia",  as_index=False)

In [ ]:
parroq.plot()

Echamos un vistazo a la tabla, que en geopandas son "geodataframes":

In [ ]:
parroq

Vemos que hay una columna denominada parroquia, con el código identificativo de cada parroquia. Otra que nos interesa contiene el nombre de cada parroquia y también otra importante es la denominada "geometry". Esta última contiene la información geográfica.

El sistema de coordenadas geográficas puede consultarse con la propiedad ```crs```:

In [ ]:
parroq.crs

Como paso previo a representar gráficamente, necesitamos unir la tabla con datos geográficos (parroq) con la tabla anterior (viv_parroq). A la hora de enlazar dos tablas, necesitamos que tengan una columna común que permita unirlas. Dicha columna es DPA_PARROQ:

In [ ]:
# Renombramos la columna parroquia como PARROQ en la geodataframe para que las columnas de la unión se llamen igual
parroq=parroq.rename(columns={"parroquia":"PARROQ"})
# Convertimos dicha columna en el mismo formato de datos
parroq.PARROQ=parroq.PARROQ.astype("int64")
# Unimos las dos tablas
parroq=parroq.merge(viv_parroq,on="PARROQ")
# Mostramos la tabla
parroq

Finalmente, creamos la representación geográfica del índice de calidad de las viviendas:

In [ ]:
parroq.plot("calidad", legend=True)

Para mejorar el aspecto del mapa, empleados el siguiente código:

In [ ]:
fig, ax=plt.subplots(dpi=100)
parroq.plot("calidad", legend=True,legend_kwds={'shrink': 0.5},ax=ax)
ax.axis("off")
ax.set_title("Índice de calidad de las viviendas en la provincia de Loja",size="11")
fig.text(0.2,0.05,"Fuente: INEC (Censo 2022). Elaboración propia.",size="8")
plt.show()

## 7. Representación geográfica (II)

Los datos nos permiten representar nuestro indicador de calidad de las viviendas con un mayor detalle geográfico, a nivel de sector censal.
Para ello, simplemente debemos seguir los mismos pasos que anteriormente, pero a nivel de sector:

1. Identificar una variable para unir la tabla de datos con la geodataframe.
2. Agrupar los datos de "vivocup" por sector.
3. Unir las tablas
4. Representar geográficamente.

In [ ]:
# 1. Código de cada sector
vivocup['sec_anm']=vivocup['ID_VIV'].astype("str").str[:12]

In [ ]:
# 2. Agrupamos por sector
columnas=['sec_anm','material', 'estado', 'salu', 'calidad']
viv_sec=vivocup[columnas].groupby(by='sec_anm', as_index=False).mean()

In [ ]:
# 3. Unimos la tabla anterior con la geodataframe
sectores=sectores.merge(viv_sec,on="sec_anm")

In [ ]:
# 4. Representamos geográficamente
fig, ax=plt.subplots(dpi=150)
sectores.plot("calidad", legend=True,legend_kwds={'shrink': 0.5},ax=ax)
parroq.boundary.plot(color="white",linewidth=0.2,ax=ax)
ax.axis("off")
ax.set_title("Índice de calidad de las viviendas en la provincia de Loja 2022",size="11")
fig.text(0.2,0.05,"Fuente: INEC (Censo 2022). Elaboración propia.",size="8")
plt.show()

In [ ]:
# Es interesante hacer zoom en la parroquia de Loja
fig, ax=plt.subplots(dpi=150)
sectores.loc[sectores.nom_par=="LOJA"].plot("calidad", legend=True,legend_kwds={'shrink': 0.5},ax=ax)
parroq.loc[parroq.nom_par=="LOJA"].boundary.plot(color="white",linewidth=0.2,ax=ax)
ax.axis("off")
ax.set_title("Índice de calidad de las viviendas en\n la parroquia de Loja 2022",size="8")
fig.text(0.4,0.05,"Fuente: INEC (Censo 2022). Elaboración propia.",size="6")
plt.show()

In [ ]:
# Y haciendo zoom en el centro...

minx, miny, maxx, maxy= parroq.loc[parroq.nom_par=="LOJA"].centroid.buffer(2500).total_bounds

fig, ax=plt.subplots(dpi=150)
sectores.loc[sectores.nom_par=="LOJA"].plot("calidad", legend=True,legend_kwds={'shrink': 0.5},ax=ax)
parroq.loc[parroq.nom_par=="LOJA"].boundary.plot(color="white",linewidth=0.2,ax=ax)
ax.axis("off")
ax.set_title("Índice de calidad de las viviendas en\n el centro la parroquia de Loja 2022\n",size="9")
ax.set_xlim(minx,maxx)
ax.set_ylim(miny,maxy)
fig.text(0.17,0.05,"Fuente: INEC (Censo 2022). Elaboración propia.",size="7")
plt.show()

## Ejercicios propuestos

1. Representar geográficamente cada uno de los tres indicadores parciales.
2. Crear un nuevo indicador que sea el porcentaje de viviendas cuyo servicio de luz (energía) eléctrica provenga principalmente de la red de empresa eléctrica de servicio público (variable V10). Calcular la media de dicha variable, mostrar su distribución de frecuencias y representar geográficamente el indicador anterior
3. Replicar el índice de calidad de las viviendas para la provincia de El Oro.

---

**Consejos básicos**:  
Al principio, la mejor manera de escribir código es reutilizando código ya escrito y adaptándolo a nuestras necesidades. Por ejemplo, los ejercicios propuestos se pueden realizar fácilmente usando el código que se proporciona en este notebook, simplemente introduciendo modificaciones mínimas.  
Hay que tener en cuenta que cualquier mínimo error al escribir el código, por pequeño que sea (falta un paréntesis, una letra aparece en mayúscula cuando debe estar en minúscula, falta una coma, una comilla, etc) nos va a dar error al ejecutar. Tenéis que ser muy cuidadosos y todo tiene que estar literalmente bien escrito.  Por ello, es importante reutilizar el código ya escrito y fijarse muy bien en que no falte nada.

---


## Recursos para el aprendizaje autónomo:

Existe una enorme cantidad de recursos disponibles en Internet para aprender a usar ```Pandas``` y más en general ```Python```.  
Por ejemplo, este curso aplicado a técnicas geoespaciales:
https://geo-python-site.readthedocs.io/en/latest/  

Hay también cursos gratuitos en plataformas como www.coursera.org o https://www.edx.org/

Es muy común consultar constantemente las dudas en Google. Lo más práctico es escribir la duda en inglés, ya que normalmente hay más ayuda. Suele ser de mucha utilidad las respuestas en el foro https://stackoverflow.com/
<br></br>
Y por supuesto, la ayuda oficial es también muy buen recurso:  
https://pandas.pydata.org/  
https://seaborn.pydata.org/  
https://matplotlib.org/

